In [1]:
%pip install anthropic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.2 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
# 1. Retrieve the key from Colab Secrets
my_anthropic_key   = userdata.get('ANTHROPIC_API_KEY')
my_anthropic_model = userdata.get('CLAUDE_MODEL')

In [3]:
# Load env variables and create client
import base64
from anthropic import Anthropic

client = Anthropic(api_key=my_anthropic_key,)
model = my_anthropic_model

In [4]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [6]:
# Define prompt
#prompt = """
#Summarize the document in one sentence
#"""

prompt = """
How were Earth's atmosphere and oceans were formed
"""

In [7]:
# TODO: Read media data, feed into Claude

def read_media_file(media_path, media_file_name):
  #with open("image.png", "rb") as f:
  #  image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
  with open(f"{media_path}/{media_file_name}", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

  return image_bytes
#


In [9]:
messages = []

image_bytes = read_media_file(".", "earth.pdf")

add_user_message(messages, [
    # Image Block
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": image_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": prompt
    }
])

chat(messages)


Message(id='msg_011Cf6fgDZTDP5X5QtWxPiiJ', container=None, content=[TextBlock(citations=None, text='# Summary\n\nEarth is the third planet from the Sun and the only known astronomical object to harbor life, sustained by its unique characteristics including liquid surface water, a dynamic atmosphere, and a protective magnetosphere.', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=9626, output_tokens=44, output_tokens_details=None, server_tool_use=None, service_tier='standard'))